# 02 — Panel Mensual Unificado

**Objetivo:** Construir el panel mensual en formato largo con columnas
`ds`, `y`, `tipo_tramite`, `sede`, `regimen`, aplicar las dos reglas de negocio
confirmadas, y guardarlo en `data/processed/panel_unificado.csv`.

## Reglas de negocio aplicadas

**Regla 1 — Exclusión de duplicados en Carné de Extranjería**
Los registros donde `TIPO_TRAMITE == 'CAMBIO DE CALIDAD MIGRATORIA'`
se solapan con el dataset independiente *Cambio de Calidad Migratoria*.
Se excluyen para evitar doble conteo.

**Regla 2 — Columna `regimen` (quiebre estructural)**
Las series de *Cambio de Calidad* y *Carné de Extranjería* muestran
una caída de ≈50% entre diciembre 2025 y enero 2026 que no es estacional.
Se marca cada fila con `pre_2026` / `post_2026` para que el modelo Prophet
pueda tratar cada período como una serie separada.
*Prórroga de Residencia* y *Solicitud de Visas* no tienen este quiebre
→ su columna `regimen` queda en `NaN`.

> 📌 Este notebook **no entrena modelos** — eso es la Fase 4 (Prophet).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Agrega la raíz del repo al path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.carga import cargar_todos_los_raw, RAW_DIR
from src.transformacion import construir_panel_mensual

PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto  : {ROOT}")
print(f"Destino del panel  : {PROCESSED_DIR}")

## 1. Carga de todos los CSV raw

In [ ]:
df_raw = cargar_todos_los_raw(anios=[2025, 2026])

print(f"Filas totales cargadas : {len(df_raw):>10,}")
print(f"Tipos de trámite       : {df_raw['tipo_tramite'].unique().tolist()}")
print(f"Años                   : {sorted(df_raw['ANIO_TRAMITE'].unique().tolist())}")
df_raw.head(3)

## 2. Construir el panel mensual unificado

La función `construir_panel_mensual` aplica internamente las dos reglas de negocio.

In [ ]:
panel = construir_panel_mensual(df_raw)

print(f"Filas en el panel : {len(panel):>8,}")
print(f"Columnas          : {list(panel.columns)}")
print(f"Rango de fechas   : {panel['ds'].min().date()} → {panel['ds'].max().date()}")
print(f"\nDistribución por tipo_tramite + regimen:")
print(
    panel.groupby(["tipo_tramite", "regimen"], dropna=False)["y"]
    .sum()
    .reset_index()
    .rename(columns={"y": "sum_y"})
    .assign(sum_y=lambda d: d["sum_y"].apply(lambda v: f"{int(v):,}"))
    .to_string(index=False)
)

In [ ]:
panel.head(10)

## 3. Verificaciones de calidad del panel

### 3.1 Sin fechas duplicadas por combinación (ds, tipo_tramite, sede, regimen)

In [ ]:
keys = ["ds", "tipo_tramite", "sede", "regimen"]
duplicados = panel[panel.duplicated(subset=keys, keep=False)]

if duplicados.empty:
    print("✅ Sin duplicados en (ds, tipo_tramite, sede, regimen)")
else:
    print(f"⚠️  {len(duplicados)} filas duplicadas detectadas:")
    display(duplicados)

### 3.2 Valores de `y` no negativos ni nulos

In [ ]:
n_nulos = panel["y"].isna().sum()
n_negativos = (panel["y"] < 0).sum()
n_cero = (panel["y"] == 0).sum()

print(f"  Nulos    en y : {n_nulos}")
print(f"  Negativos en y: {n_negativos}")
print(f"  Ceros    en y : {n_cero}  (podrían ser meses sin actividad en cierta sede)")

if n_nulos == 0 and n_negativos == 0:
    print("\n✅ Columna 'y' sin valores nulos ni negativos")

### 3.3 Cobertura temporal por tipo de trámite

In [ ]:
from IPython.display import display

cobertura = (
    panel.groupby("tipo_tramite")["ds"]
    .agg(primer_mes="min", ultimo_mes="max", n_meses_sede="count")
    .reset_index()
)
# Meses únicos (independiente de sede)
meses_unicos = (
    panel.groupby("tipo_tramite")["ds"]
    .nunique()
    .reset_index()
    .rename(columns={"ds": "meses_unicos"})
)
cobertura = cobertura.merge(meses_unicos, on="tipo_tramite")
display(cobertura)

### 3.4 Verificación de la Regla 1: no hay TIPO_TRAMITE='CAMBIO DE CALIDAD MIGRATORIA' en carnet_extranjeria

In [ ]:
# El panel ya está agregado (no tiene TIPO_TRAMITE), así que verificamos
# la regla en el DataFrame raw antes de transformar:
from src.transformacion import aplicar_regla_carnet

# Tomar solo el carnet del raw
df_carnet_raw = df_raw[df_raw["tipo_tramite"] == "carnet_extranjeria"].copy()
df_carnet_limpio = aplicar_regla_carnet(df_raw)  # Aplica filtro
df_carnet_limpio = df_carnet_limpio[df_carnet_limpio["tipo_tramite"] == "carnet_extranjeria"]

n_antes = len(df_carnet_raw)
n_despues = len(df_carnet_limpio)
print(f"Filas Carné (antes regla)  : {n_antes:>7,}")
print(f"Filas Carné (después regla): {n_despues:>7,}")
print(f"Filas excluidas            : {n_antes - n_despues:>7,}")

# Confirmar que en el resultado ya no queda ningún solapo
solapo_residual = df_carnet_limpio[
    df_carnet_limpio["TIPO_TRAMITE"].str.strip().str.upper() == "CAMBIO DE CALIDAD MIGRATORIA"
]
if solapo_residual.empty:
    print("✅ Regla 1 verificada: sin solapamiento residual en Carné")
else:
    print(f"⚠️  {len(solapo_residual)} filas de solapamiento residual")

### 3.5 Verificación de la Regla 2: columna `regimen` correctamente asignada

In [ ]:
TRAMITES_CON_REGIMEN = {"cambio_calidad", "carnet_extranjeria"}
TRAMITES_SIN_REGIMEN = {"prorroga_residencia", "solicitud_visas"}

for tipo in TRAMITES_CON_REGIMEN:
    valores = panel.loc[panel["tipo_tramite"] == tipo, "regimen"].unique()
    ok = set(valores) == {"pre_2026", "post_2026"}
    marca = "✅" if ok else "⚠️"
    print(f"  {marca} {tipo}: regimen = {sorted(valores)}")

for tipo in TRAMITES_SIN_REGIMEN:
    valores = panel.loc[panel["tipo_tramite"] == tipo, "regimen"].dropna().unique()
    ok = len(valores) == 0
    marca = "✅" if ok else "⚠️"
    print(f"  {marca} {tipo}: regimen = NaN (correcto)" if ok else f"  ⚠️ {tipo}: valores inesperados = {valores}")

## 4. Vista de serie de tiempo agregada nacional (totales por mes)

Gráfico de tendencia para confirmar visualmente la calidad del panel.

In [ ]:
COLORES = {
    "prorroga_residencia": "#2196F3",
    "carnet_extranjeria": "#FF9800",
    "cambio_calidad": "#4CAF50",
    "solicitud_visas": "#9C27B0",
}

ETIQUETAS = {
    "prorroga_residencia": "Prórroga de Residencia",
    "carnet_extranjeria": "Carné de Extranjería (neto)",
    "cambio_calidad": "Cambio de Calidad Migratoria",
    "solicitud_visas": "Solicitud de Calidad (Visas)",
}

serie_nacional = (
    panel.groupby(["ds", "tipo_tramite"])["y"]
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for ax, tipo in zip(axes, ["prorroga_residencia", "carnet_extranjeria", "cambio_calidad", "solicitud_visas"]):
    sub = serie_nacional[serie_nacional["tipo_tramite"] == tipo].sort_values("ds")
    ax.plot(sub["ds"], sub["y"], marker="o", color=COLORES[tipo], linewidth=2, markersize=4)
    ax.axvline(pd.Timestamp("2026-01-01"), color="red", linestyle="--", alpha=0.6, linewidth=1.2)
    ax.set_title(ETIQUETAS[tipo], fontsize=10, pad=4)
    ax.set_ylabel("Cantidad")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(alpha=0.25)

plt.suptitle(
    "Panel Mensual Unificado — Serie nacional agregada\n(línea roja = enero 2026)",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

## 5. Top 5 sedes por volumen total

In [ ]:
top_sedes = (
    panel.groupby("sede")["y"]
    .sum()
    .nlargest(5)
    .reset_index()
    .rename(columns={"y": "total_tramites"})
)
top_sedes["total_tramites"] = top_sedes["total_tramites"].apply(lambda x: f"{int(x):,}")
top_sedes

## 6. Guardar panel en data/processed/

In [ ]:
RUTA_SALIDA = PROCESSED_DIR / "panel_unificado.csv"

panel.to_csv(RUTA_SALIDA, index=False, date_format="%Y-%m-%d")

tam_kb = RUTA_SALIDA.stat().st_size / 1024
print(f"✅ Panel guardado en: {RUTA_SALIDA}")
print(f"   Tamaño           : {tam_kb:.1f} KB")
print(f"   Filas            : {len(panel):,}")
print(f"   Columnas         : {list(panel.columns)}")

## 7. Verificación final: recargar desde disco

In [ ]:
panel_verificado = pd.read_csv(RUTA_SALIDA, parse_dates=["ds"])

assert len(panel_verificado) == len(panel), "⚠️ Discrepancia en número de filas"
assert list(panel_verificado.columns) == ["ds", "y", "tipo_tramite", "sede", "regimen"], "⚠️ Columnas incorrectas"
assert panel_verificado["y"].isna().sum() == 0, "⚠️ Nulos en y"

print("✅ Verificación final aprobada")
print(f"   Rango ds    : {panel_verificado['ds'].min().date()} → {panel_verificado['ds'].max().date()}")
print(f"   sum(y) total: {panel_verificado['y'].sum():,}")
panel_verificado.dtypes

---
**El panel `data/processed/panel_unificado.csv` está listo para la Fase 4 (Prophet).**

Columnas del panel:
| Columna | Tipo | Descripción |
|---|---|---|
| `ds` | date | Primer día del mes |
| `y` | int | Suma de CANTIDAD |
| `tipo_tramite` | str | Clave corta del trámite |
| `sede` | str | SEDE_ATENCION |
| `regimen` | str/NaN | `pre_2026` / `post_2026` / NaN |